# Multi-Vintage Mortgage Data Ingestion


Extend the corrected 2019 target-construction workflow across Freddie Mac origination vintages from 2015 through 2022.


## 1. Setup


In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_processing import build_model_data, build_target

DATA_DIR = Path("../data/raw_mortgage")
YEARS = list(range(2015, 2023))

ImportError: cannot import name 'build_model_data' from 'src.data_processing' (/Users/noahlatman/Downloads/ML Projects/Mortgage Risk Project/src/data_processing.py)

In [2]:
for year in YEARS:
    orig_path = DATA_DIR / f"sample_{year}" / f"sample_orig_{year}.txt"
    perf_path = DATA_DIR / f"sample_{year}" / f"sample_perf_{year}.txt"
    print(year, "| Orig:", orig_path.exists(), "| Perf:", perf_path.exists())


2015 | Orig: True | Perf: True
2016 | Orig: True | Perf: True
2017 | Orig: True | Perf: True
2018 | Orig: True | Perf: True
2019 | Orig: True | Perf: True
2020 | Orig: True | Perf: True
2021 | Orig: True | Perf: True
2022 | Orig: True | Perf: True


## 2. Dataset Schema


In [3]:
orig_columns = [
    "credit_score",
    "first_payment_date",
    "first_time_homebuyer",
    "maturity_date",
    "msa",
    "mi_percent",
    "num_units",
    "occupancy_status",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "original_interest_rate",
    "channel",
    "prepayment_penalty",
    "amortization_type",
    "property_state",
    "property_type",
    "postal_code",
    "loan_id",
    "loan_purpose",
    "original_loan_term",
    "num_borrowers",
    "seller_name",
    "super_conforming_flag",
    "pre_harp_loan_id",
    "special_eligibility_program",
    "harp_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "vantage_score"
]


## 3. Reusable Data Loaders


In [4]:
def load_origination(year):
    path = DATA_DIR / f"sample_{year}" / f"sample_orig_{year}.txt"
    df = pd.read_csv(path, sep="|", header=None, names=orig_columns, low_memory=False)
    df["origination_vintage"] = year
    return df


In [5]:
def load_performance(year):
    path = DATA_DIR / f"sample_{year}" / f"sample_perf_{year}.txt"
    df = pd.read_csv(path, sep="|", header=None, usecols=[0, 1, 3, 4, 8], low_memory=False)
    df.columns = ["loan_id", "monthly_reporting_period", "delinquency_status", "loan_age", "zero_balance_code"]
    return df


## 4. Calendar-Based Target Construction


Use elapsed calendar time from the first-payment month rather than Freddie Mac's reported `loan_age`, which can reset later in a loan's history.


In [6]:
def build_target(year):
    perf = load_performance(year)
    orig_dates = load_origination(year)[["loan_id", "first_payment_date"]].copy()

    perf["reporting_month"] = pd.to_datetime(
        perf["monthly_reporting_period"].astype(str), format="%Y%m"
    )
    orig_dates["first_payment_month"] = pd.to_datetime(
        orig_dates["first_payment_date"].astype(str), format="%Y%m"
    )

    perf = perf.merge(
        orig_dates[["loan_id", "first_payment_month"]],
        on="loan_id", how="left", validate="many_to_one"
    )

    perf["calendar_loan_age"] = (
        (perf["reporting_month"].dt.year - perf["first_payment_month"].dt.year) * 12
        + (perf["reporting_month"].dt.month - perf["first_payment_month"].dt.month)
        + 1
    )
    perf["delinquency_numeric"] = pd.to_numeric(perf["delinquency_status"], errors="coerce")

    perf_sorted = perf.sort_values(["loan_id", "reporting_month"])
    loan_followup = (
        perf_sorted.groupby("loan_id", as_index=False)
        .agg(max_calendar_age=("calendar_loan_age", "max"))
    )
    terminal_rows = (
        perf_sorted.groupby("loan_id", as_index=False)
        .nth(-1)[["loan_id", "zero_balance_code"]]
        .rename(columns={"zero_balance_code": "last_zero_balance_code"})
    )
    loan_followup = loan_followup.merge(
        terminal_rows, on="loan_id", how="left", validate="one_to_one"
    )
    early_defect_ids = set(
        loan_followup.loc[
            (loan_followup["max_calendar_age"] < 24)
            & (loan_followup["last_zero_balance_code"] == 96),
            "loan_id"
        ]
    )

    perf_24m = perf[perf["calendar_loan_age"].between(0, 24)].copy()
    target = (
        perf_24m.groupby("loan_id", as_index=False)
        .agg(max_delinquency_24m=("delinquency_numeric", "max"))
    )
    target["seriously_delinquent_24m"] = target["max_delinquency_24m"] >= 3
    return target[~target["loan_id"].isin(early_defect_ids)].copy()


### 4.1 Validate Against 2019


In [7]:
target_2019 = build_target(2019)
print("Loans:", len(target_2019))
print("Serious delinquencies:", target_2019["seriously_delinquent_24m"].sum())
print("Rate:", target_2019["seriously_delinquent_24m"].mean())


Loans: 49897
Serious delinquencies: 2285
Rate: 0.04579433633284566


**Observation:** The reusable calendar-based target should reproduce the canonical 2019 population of 49,897 loans with 2,285 serious-delinquency events.


## 5. Build Multi-Vintage Modeling Data


In [8]:
def clean_origination(orig):
    missing_codes = {
        "credit_score": [9999, "9999"], "mi_percent": [999, "999"],
        "num_units": [99, "99"], "occupancy_status": [9, "9"],
        "cltv": [999, "999"], "dti": [999, "999"], "ltv": [999, "999"],
        "channel": [9, "9"], "property_type": [99, "99"],
        "postal_code": [0, "000"], "loan_purpose": [9, "9"],
        "num_borrowers": [99, "99"], "property_valuation_method": [7, "7"],
        "vantage_score": [9999, "9999"]
    }
    for col, sentinels in missing_codes.items():
        orig[col] = orig[col].replace(sentinels, pd.NA)
    return orig


In [9]:
def build_model_data(year):
    orig = clean_origination(load_origination(year))
    target = build_target(year)
    df = orig.merge(
        target[["loan_id", "seriously_delinquent_24m"]],
        on="loan_id", how="inner", validate="one_to_one"
    )
    return df.drop(columns=["vantage_score", "pre_harp_loan_id"], errors="ignore")


### 5.1 Process All Vintages


In [10]:
vintage_datasets = []
vintage_summary = []

for year in YEARS:
    df = build_model_data(year)
    vintage_datasets.append(df)
    vintage_summary.append({
        "vintage": year,
        "loans": len(df),
        "serious_delinquencies": int(df["seriously_delinquent_24m"].sum()),
        "delinquency_rate": df["seriously_delinquent_24m"].mean()
    })

vintage_summary = pd.DataFrame(vintage_summary)
vintage_summary


,vintage,loans,serious_delinquencies,delinquency_rate
0,2015,48626,250,0.005141
1,2016,49613,329,0.006631
2,2017,49841,395,0.007925
3,2018,49908,1274,0.025527
4,2019,49897,2285,0.045794
5,2020,49877,791,0.015859
6,2021,49895,445,0.008919
7,2022,49766,882,0.017723


In [11]:
multi_vintage_data = pd.concat(vintage_datasets, ignore_index=True)
print("Shape:", multi_vintage_data.shape)
print("Unique loans:", multi_vintage_data["loan_id"].nunique())
print("Vintages:", sorted(multi_vintage_data["origination_vintage"].unique()))


Shape: (397423, 31)
Unique loans: 397423
Vintages: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


**Observation:** The corrected 2015–2022 pipeline should produce 397,423 unique mortgages. Vintage-level event rates vary substantially, motivating out-of-time validation rather than a random pooled split.


## 6. Save Processed Dataset


In [12]:
output_path = "../data/processed/model_data_2015_2022.parquet"
multi_vintage_data.to_parquet(output_path, index=False)
print("Saved:", output_path)


Saved: ../data/processed/model_data_2015_2022.parquet


In [13]:
check = pd.read_parquet(output_path)
print("Shape:", check.shape)
print("Unique loans:", check["loan_id"].nunique())
print("Vintages:", sorted(check["origination_vintage"].unique()))
print()
print(check.groupby("origination_vintage")["seriously_delinquent_24m"].agg(["count", "sum", "mean"]))


Shape: (397423, 31)
Unique loans: 397423
Vintages: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]

                     count   sum      mean
origination_vintage                       
2015                 48626   250  0.005141
2016                 49613   329  0.006631
2017                 49841   395  0.007925
2018                 49908  1274  0.025527
2019                 49897  2285  0.045794
2020                 49877   791  0.015859
2021                 49895   445  0.008919
2022                 49766   882  0.017723
